# RAG

In [1]:
!pip install -q pandas numpy sentence-transformers faiss-cpu transformers accelerate gradio


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Chargement & nettoyage

In [2]:
import pandas as pd
import numpy as np
import re, unicodedata, warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('export_final.csv', encoding='utf-8-sig')
df = df.dropna(subset=['infraction_desc'])

def clean(text):
    if not isinstance(text, str): return ''
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'[\u200b-\u200d\ufeff]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['infraction_desc'].apply(clean)
df = df[df['text'].str.len() > 30].reset_index(drop=True)
print(f'✅ {len(df)} articles chargés')

✅ 317 articles chargés


## 2. Chunking

In [3]:
chunks = []
for _, row in df.iterrows():
    text = row['text']
    size, overlap = 250, 50
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        chunks.append({
            'article_id'    : int(row['article_id']),
            'type_sanction' : str(row.get('type_sanction', '')),
            'amende_min'    : row.get('amende_min', ''),
            'amende_max'    : row.get('amende_max', ''),
            'points_retrait': row.get('points_retrait', ''),
            'text'          : text[start:end]
        })
        if end == len(text): break
        start += size - overlap

print(f'✅ {len(chunks)} chunks créés')

✅ 547 chunks créés


## 3. Embeddings + FAISS

In [4]:
import faiss
from sentence_transformers import SentenceTransformer

print('Chargement embedder...')
embedder = SentenceTransformer('intfloat/multilingual-e5-small')  # ~120 MB

emb = embedder.encode(
    [c['text'] for c in chunks],
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
).astype(np.float32)

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print(f'✅ FAISS : {index.ntotal} vecteurs')

Chargement embedder...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1637.08it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 9/9 [00:42<00:00,  4.73s/it]

✅ FAISS : 547 vecteurs


## 4. LLM

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = 'Qwen/Qwen2-0.5B-Instruct'
print(f'Chargement {MODEL} ...')

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float32
)
model.eval()
print('✅ Modèle chargé')

Chargement Qwen/Qwen2-0.5B-Instruct ...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:12<00:00, 23.15it/s]


✅ Modèle chargé


## 5. Pipeline RAG strict

In [6]:
OOD_THRESHOLD = 0.42
OOD_MSG = '⚠️ هذا السؤال خارج نطاق قانون السير المغربي. لا أستطيع الإجابة عليه.'
NO_INFO_MSG = '⚠️ لا تتوفر معلومات كافية في قاعدة البيانات للإجابة على هذا السؤال.'


def retrieve(query, k=4):
    q = embedder.encode([clean(query)], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q, k)
    return [
        {**chunks[i], 'score': float(s)}
        for s, i in zip(scores[0], ids[0]) if i >= 0
    ]


def is_ood(query):
    r = retrieve(query, k=1)
    return (not r) or r[0]['score'] < OOD_THRESHOLD


def build_messages(question, docs):
    # Contexte court = génération plus rapide
    context = ''
    for d in docs:
        line = f"[المادة {d['article_id']}]"
        if str(d['type_sanction']) not in ('non_defini','nan','','None'):
            line += f" عقوبة:{d['type_sanction']}"
        if str(d['amende_min']) not in ('nan','','None'):
            line += f" غرامة:{d['amende_min']}-{d['amende_max']}DH"
        if str(d['points_retrait']) not in ('nan','','None'):
            line += f" نقاط:{d['points_retrait']}"
        context += f"{line}\n{d['text']}\n\n"

    system = (
        "أنت مساعد قانوني. أجب فقط من المواد المقدمة. "
        "إذا لم تجد الإجابة اكتب بالضبط: لا تتوفر معلومات كافية. "
        "اذكر رقم المادة دائماً. لا تخترع معلومات."
    )
    user = f"المواد:\n{context}\nالسؤال: {question}\nالجواب:"
    return [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': user}
    ]


def generate(messages, max_new_tokens=150):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors='pt')

    # Limiter le contexte pour accélérer la génération
    if inputs['input_ids'].shape[1] > 1200:
        inputs = {k: v[:, -1200:] for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    new = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()


def rag(question):
    question = question.strip()
    if not question:
        return 'الرجاء إدخال سؤال.', '', ''

    if is_ood(question):
        return OOD_MSG, '', ''

    docs     = retrieve(question, k=4)
    messages = build_messages(question, docs)
    answer   = generate(messages)

    # Filet anti-hallucination
    if 'المادة' not in answer and len(answer) > 40:
        answer = NO_INFO_MSG

    refs = '، '.join(sorted(set(f"المادة {d['article_id']}" for d in docs)))

    previews = ''
    for d in docs:
        tag = f"📌 المادة {d['article_id']} (تطابق: {d['score']:.2f})"
        if str(d['amende_min']) not in ('nan','','None'):
            tag += f" | {d['amende_min']}-{d['amende_max']} DH"
        previews += f"{tag}\n{d['text'][:200]}...\n\n"

    return answer, refs, previews


# ── Tests rapides ─────────────────────────────────────────────────────────────
print('=== Test في الدومين ===')
a, r, _ = rag('ما هي عقوبة القيادة بدون رخصة؟')
print('الجواب:', a)
print('المراجع:', r)

print()
print('=== Test hors domaine ===')
a2, _, _ = rag('كيف أطبخ الكسكس')
print('الجواب:', a2)

=== Test في الدومين ===


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


الجواب: ⚠️ لا تتوفر معلومات كافية في قاعدة البيانات للإجابة على هذا السؤال.
المراجع: المادة 141، المادة 28، المادة 293، المادة 299

=== Test hors domaine ===
الجواب: ⚠️ لا تتوفر معلومات كافية في قاعدة البيانات للإجابة على هذا السؤال.


## 6. Interface Gradio

In [7]:
import gradio as gr

css = """
.rtl textarea, .rtl label {
    direction: rtl;
    text-align: right;
    font-family: Arial, sans-serif;
    font-size: 15px;
}
#title { text-align: center; }
"""

with gr.Blocks(title='مساعد قانون السير', theme=gr.themes.Soft(), css=css) as demo:

    gr.Markdown('# 🚦 مساعد قانون السير المغربي', elem_id='title')
    gr.Markdown(
        'يجيب هذا المساعد **فقط** بناءً على مواد قانون السير.  '
        'إذا لم تتوفر الإجابة، سيخبرك بذلك صراحةً.'
    )

    with gr.Row():
        q_in = gr.Textbox(
            label='سؤالك',
            placeholder='مثال: ما هي عقوبة تجاوز السرعة؟',
            lines=2, scale=4,
            elem_classes=['rtl']
        )
        btn = gr.Button('🔍 بحث', variant='primary', scale=1)

    ans_out  = gr.Textbox(label='الجواب',  lines=5,
                          elem_classes=['rtl'], interactive=False)
    refs_out = gr.Textbox(label='المواد المستخدمة', lines=1,
                          elem_classes=['rtl'], interactive=False)

    with gr.Accordion('📂 الوثائق المستردة', open=False):
        docs_out = gr.Textbox(label='نصوص المواد', lines=8,
                              elem_classes=['rtl'], interactive=False)

    gr.Examples(
        examples=[
            ['ما هي عقوبة القيادة بدون رخصة؟'],
            ['ما هي شروط الحصول على رخصة السياقة؟'],
            ['كم نقطة تُسحب عند ارتكاب مخالفة؟'],
            ['ما التزامات الفحص الطبي لحاملي الرخصة؟'],
            ['ما هي حقوق السائق الأجنبي في المغرب؟'],
            ['ما هو سعر الذهب اليوم'],
        ],
        inputs=[q_in]
    )

    btn.click(fn=rag, inputs=q_in, outputs=[ans_out, refs_out, docs_out])
    q_in.submit(fn=rag, inputs=q_in, outputs=[ans_out, refs_out, docs_out])

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://0bce8ee15257636807.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
